In [ ]:
# PANAMA ONE HEALTH DISTRICT DATAFRAME

# Geographic standard: ANATI 2025 district boundaries
# Output: one row per ANATI district

# Predictors:
#   1. Health facilities per 10,000 population - pop size should be another predictor
#   2. Dominant land-cover class
#   3. WaPOR AETI trend, 2021-2023
#   4. Purchasing power
#   5. Protected area percentage

# Outcome:
#   Mortality rate, 2021-2023

import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import re
import unicodedata
import json

from pathlib import Path

ee.Authenticate()
ee.Initialize()

### Input files

In [1]:
# Input files

POPULATION_FILE = "panama_population.csv"
PURCHASING_POWER_FILE = "panama_purchasing_power.csv"
MORTALITY_FILE = "panama_mortality.csv"


# Output files

OUTPUT_CSV = "panama_one_health_districts.csv"
OUTPUT_GPKG = "panama_one_health_districts.gpkg"

### Analysis settings

In [2]:
# Analysis settings

ANALYSIS_YEARS = [2021, 2022, 2023]

# Assumed unit for published INEC mortality rates.
# Change to 100000 if rates are published per 100,000.
MORTALITY_RATE_MULTIPLIER = 1

# WaPOR AETI scale factor used in the earlier workflow.
# Confirm against the collection's band metadata for your version.
WAPOR_SCALE_FACTOR = 0.1

# WaPOR L1 AETI nominal analysis scale.
WAPOR_SCALE_METERS = 250

# SINIA land-cover nominal analysis scale.
LULC_SCALE_METERS = 30

# Protected-area geometry error margin.
GEOMETRY_ERROR_METERS = 10

# Set to a list of actual TIPO values if you want to exclude
# records that are not health facilities.
#
# Leave as None initially to include all records, then inspect
# the TIPO frequency table printed below.
FACILITY_TYPES_TO_INCLUDE = None

### Helper functions

In [3]:
# 1. HELPER FUNCTIONS

def normalize_name(value):
    """
    Normalize administrative names for matching.

    Removes accents, punctuation, extra spaces, and capitalization
    differences.
    """

    if pd.isna(value):
        return ""

    value = str(value).strip().upper()

    value = unicodedata.normalize(
        "NFKD", value
    ).encode("ASCII", "ignore").decode("utf-8")

    value = re.sub(r"[^A-Z0-9 ]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value


def add_admin_key(data):
    """
    Build the common administrative join key.
    """

    data = data.copy()

    data["ADMIN_KEY"] = (
        data["PROVINCE"].apply(normalize_name)
        + "|"
        + data["DISTRICT"].apply(normalize_name)
    )

    return data


def check_unique_admin_keys(data, label):
    """
    Prevent accidental many-to-many joins.
    """

    duplicates = data[
        data["ADMIN_KEY"].duplicated(keep=False)
    ].sort_values("ADMIN_KEY")

    if not duplicates.empty:
        print(f"\nDuplicate administrative keys in {label}:")
        print(
            duplicates[
                ["PROVINCE", "DISTRICT", "ADMIN_KEY"]
            ].to_string(index=False)
        )

        raise ValueError(
            f"{label} contains duplicate district keys. "
            "Resolve these before merging."
        )


def read_admin_csv(path, label):
    """
    Read an external administrative CSV and standardize its keys.
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Prepare this CSV before running."
        )

    data = pd.read_csv(path)

    required = ["DISTRICT", "PROVINCE"]

    missing = [
        column for column in required
        if column not in data.columns
    ]

    if missing:
        raise ValueError(
            f"{label} is missing required columns: {missing}"
        )

    data = add_admin_key(data)

    check_unique_admin_keys(data, label)

    return data


def report_unmatched(source, standard, label):
    """
    Report records that do not match the ANATI district standard.
    """

    standard_keys = set(standard["ADMIN_KEY"])

    unmatched = source[
        ~source["ADMIN_KEY"].isin(standard_keys)
    ]

    print(f"\n{label}: unmatched source districts = {len(unmatched)}")

    if not unmatched.empty:
        print(
            unmatched[
                ["PROVINCE", "DISTRICT", "ADMIN_KEY"]
            ].to_string(index=False)
        )


def safe_float(value):
    """
    Convert values to floats, returning NaN for invalid values.
    """

    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

### ANATI district boundaries

In [ ]:
# 2. DOWNLOAD ANATI DISTRICT BOUNDARIES

print("\nDownloading ANATI district boundaries...")


district_url = (
    "https://sigigntg.anati.gob.pa/arcgisserver/rest/services/"
    "Mapa_Web_Divisi%C3%B3n_Pol%C3%ADtica_Administrativa_2025a_MIL1/"
    "MapServer/7/query"
)

district_params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson"
}

response = requests.get(
    district_url,
    params=district_params,
    timeout=120
)

response.raise_for_status()

district_geojson = response.json()

if "features" not in district_geojson:
    raise ValueError(
        f"ANATI boundary service did not return features: "
        f"{district_geojson}"
    )

districts_gdf = gpd.GeoDataFrame.from_features(
    district_geojson["features"],
    crs="EPSG:4326"
)

print("Fields returned by ANATI:")
print(districts_gdf.columns.tolist())

print("Downloaded boundary features:", len(districts_gdf))


# Verify required ANATI fields

required_boundary_fields = [
    "nomb_dist",
    "nomb_prov",
    "cod_dist"
]

missing_fields = [
    field for field in required_boundary_fields
    if field not in districts_gdf.columns
]

if missing_fields:
    raise ValueError(
        f"ANATI dataset is missing expected fields: {missing_fields}"
    )


# Standardize ANATI fields

districts_gdf["DISTRICT"] = (
    districts_gdf["nomb_dist"].astype(str).str.strip()
)

districts_gdf["PROVINCE"] = (
    districts_gdf["nomb_prov"].astype(str).str.strip()
)

districts_gdf["DISTRICT_CODE"] = (
    districts_gdf["cod_dist"].astype(str).str.strip()
)

districts_gdf = districts_gdf[
    districts_gdf.geometry.notna()
].copy()

districts_gdf = districts_gdf[
    ~districts_gdf.geometry.is_empty
].copy()


# Inspect administrative units

print("\nANATI administrative diagnostics")

print(
    "Unique district codes:",
    districts_gdf["DISTRICT_CODE"].nunique()
)

print(
    "Unique district names:",
    districts_gdf["DISTRICT"].nunique()
)

print("\nUnique district codes by province:")

print(
    districts_gdf.groupby("PROVINCE")[
        "DISTRICT_CODE"
    ].nunique()
)


# Dissolve multiple features
# representing the same district

# A district may be represented by more than one polygon feature.
# Dissolve those features using province + district code + name.
#
# Do not force the result to a presumed number of districts.

districts_gdf = districts_gdf.dissolve(
    by=[
        "PROVINCE",
        "DISTRICT_CODE",
        "DISTRICT"
    ],
    as_index=False
)

districts_gdf = districts_gdf[
    [
        "DISTRICT_CODE",
        "DISTRICT",
        "PROVINCE",
        "geometry"
    ]
].copy()


# Repair invalid polygon geometries if necessary.
# buffer(0) is a common fallback for minor topology issues.
invalid_before = (~districts_gdf.geometry.is_valid).sum()

print("\nInvalid geometries before repair:", invalid_before)

if invalid_before > 0:
    districts_gdf["geometry"] = (
        districts_gdf.geometry.buffer(0)
    )

invalid_after = (~districts_gdf.geometry.is_valid).sum()

print("Invalid geometries after repair:", invalid_after)

if invalid_after > 0:
    raise ValueError(
        "Some ANATI geometries remain invalid. "
        "Repair or inspect them before spatial analysis."
    )


# Build the permanent join key

districts_gdf = add_admin_key(districts_gdf)

check_unique_admin_keys(
    districts_gdf,
    "ANATI district standard"
)

districts_gdf = districts_gdf.sort_values(
    ["PROVINCE", "DISTRICT"]
).reset_index(drop=True)

print("\nFinal ANATI district count:", len(districts_gdf))

print(
    districts_gdf[
        ["DISTRICT_CODE", "PROVINCE", "DISTRICT"]
    ].to_string(index=False)
)


# Convert ANATI boundaries to GEE

districts_ee = geemap.geopandas_to_ee(
    districts_gdf
)

panama_geom = districts_ee.geometry()

NameError: name 'requests' is not defined

### Health facilities

In [ ]:
# 3. HEALTH FACILITIES: STRI / ARCGIS

print("\nDownloading health facility records...")


health_url = (
    "https://services2.arcgis.com/HRY6x8qt5qjGnAA9/"
    "ArcGIS/rest/services/Instalaciones_de_Salud/"
    "FeatureServer/0/query"
)


# Get all object IDs first

id_response = requests.get(
    health_url,
    params={
        "where": "1=1",
        "returnIdsOnly": "true",
        "f": "json"
    },
    timeout=120
)

id_response.raise_for_status()

id_data = id_response.json()

if "objectIds" not in id_data:
    raise ValueError(
        f"Could not retrieve health facility IDs: {id_data}"
    )

object_ids = id_data["objectIds"]

print("Health facility IDs returned:", len(object_ids))


# Download features in chunks
# This avoids silent ArcGIS record limits.

health_features = []

CHUNK_SIZE = 500

for start in range(0, len(object_ids), CHUNK_SIZE):

    ids_chunk = object_ids[
        start:start + CHUNK_SIZE
    ]

    feature_response = requests.get(
        health_url,
        params={
            "objectIds": ",".join(map(str, ids_chunk)),
            "outFields": "*",
            "returnGeometry": "true",
            "outSR": "4326",
            "f": "geojson"
        },
        timeout=120
    )

    feature_response.raise_for_status()

    feature_data = feature_response.json()

    if "features" not in feature_data:
        raise ValueError(
            f"Invalid facility response: {feature_data}"
        )

    health_features.extend(feature_data["features"])


health_gdf = gpd.GeoDataFrame.from_features(
    health_features,
    crs="EPSG:4326"
)

print("Health facility records downloaded:", len(health_gdf))


if len(health_gdf) != len(object_ids):
    print(
        "WARNING: downloaded feature count differs from "
        "the object ID count. Check the ArcGIS response."
    )


# Inspect facility categories

print("\nHealth facility fields:")
print(health_gdf.columns.tolist())

if "TIPO" in health_gdf.columns:

    print("\nFacility type counts:")

    print(
        health_gdf["TIPO"].value_counts(
            dropna=False
        )
    )

else:
    print(
        "WARNING: no TIPO field found. "
        "All downloaded records will be treated as facilities."
    )


# Optional facility type filter

if FACILITY_TYPES_TO_INCLUDE is not None:

    if "TIPO" not in health_gdf.columns:
        raise ValueError(
            "A facility type filter was specified, "
            "but no TIPO field exists."
        )

    health_gdf = health_gdf[
        health_gdf["TIPO"].isin(
            FACILITY_TYPES_TO_INCLUDE
        )
    ].copy()


# Retain valid point geometries

health_gdf = health_gdf[
    health_gdf.geometry.notna()
].copy()

health_gdf = health_gdf[
    ~health_gdf.geometry.is_empty
].copy()

health_gdf = health_gdf[
    health_gdf.geometry.geom_type == "Point"
].copy()


# Convert facilities to GEE

health_facilities_ee = geemap.geopandas_to_ee(
    health_gdf
)


# Count facilities in each ANATI district

def count_health_facilities(district):

    district_geom = district.geometry()

    facility_count = (
        health_facilities_ee
        .filterBounds(district_geom)
        .size()
    )

    district_area_km2 = (
        district_geom.area(GEOMETRY_ERROR_METERS)
        .divide(1e6)
    )

    return district.set({
        "HEALTH_FACILITY_COUNT": facility_count,
        "DISTRICT_AREA_KM2": district_area_km2
    })


districts_ee = districts_ee.map(
    count_health_facilities
)

### Land use land cover

In [ ]:
# 4. LAND COVER: SINIA 2021

print("\nProcessing SINIA land cover...")


# Load SINIA image

sinia_image = ee.Image(
    "projects/deforestation-panama/assets/landuse_2021"
).clip(panama_geom)


# Reclassification supplied
# in the earlier workflow

from_list = list(range(1, 34))

to_list = [
    10, 20, 80, 80, 80, 80, 20, 20,
    60, 60, 60, 70, 70, 50, 40, 50,
    40, 40, 40, 50, 40, 40, 40, 40,
    50, 60, 27, 28, 29, 70, 70, 70, 70
]


class_labels = {
    10: "Mature forest",
    20: "Secondary forest",
    30: "Primary forest",
    40: "Crops",
    50: "Plantation",
    60: "Other vegetation",
    70: "Other landuse",
    80: "Other forest",
    27: "Water surface",
    28: "Populated area",
    29: "Infrastructure"
}


lulc = (
    sinia_image
    .remap(from_list, to_list)
    .rename("landuse")
)

# Calculate land-cover
# histogram per district

def calculate_lulc(district):

    histogram = lulc.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=district.geometry(),
        scale=LULC_SCALE_METERS,
        maxPixels=1e10,
        tileScale=4
    ).get("landuse")

    return district.set(
        "LULC_HISTOGRAM",
        histogram
    )


districts_ee = districts_ee.map(
    calculate_lulc
)

### Evapotranspiration

In [ ]:
# 5. WaPOR AETI: 2021-2023

print("\nProcessing WaPOR AETI...")


wapor_collection = (
    ee.ImageCollection("FAO/WAPOR/3/L1_AETI_D")
    .filterDate("2021-01-01", "2024-01-01")
    .filterBounds(panama_geom)
    .select("L1-AETI-D")
)


# Check that each year has data

yearly_aeti_images = []

for year in ANALYSIS_YEARS:

    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = start_date.advance(1, "year")

    yearly_collection = wapor_collection.filterDate(
        start_date,
        end_date
    )

    image_count = yearly_collection.size().getInfo()

    print(f"WaPOR images in {year}: {image_count}")

    if image_count == 0:
        raise ValueError(
            f"No WaPOR AETI images found for {year}."
        )

    annual_image = (
        yearly_collection
        .sum()
        .multiply(WAPOR_SCALE_FACTOR)
        .rename(f"AETI_{year}")
        .float()
    )

    yearly_aeti_images.append(annual_image)


# Combine annual images into one multiband image.
annual_aeti = ee.Image.cat(
    yearly_aeti_images
)


# Calculate annual district means

def calculate_aeti(district):

    annual_means = annual_aeti.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=district.geometry(),
        scale=WAPOR_SCALE_METERS,
        maxPixels=1e10,
        tileScale=4
    )

    return district.set(annual_means)


districts_ee = districts_ee.map(
    calculate_aeti
)

WDPA protected areas

In [ ]:
# 6. PROTECTED AREAS: WDPA

print("\nProcessing protected areas...")


# IMPORTANT:
# This is the current WDPA snapshot, not a historical 2021 snapshot.
# Use an archived WDPA release if you need protected-area status
# specifically as of 2021.

panama_pas = (
    ee.FeatureCollection("WCMC/WDPA/current/polygons")
    .filter(ee.Filter.eq("ISO3", "PAN"))
    .filterBounds(panama_geom)
)


print(
    "WDPA protected-area features:",
    panama_pas.size().getInfo()
)


# Calculate polygon overlap with each ANATI district

def calculate_protected_area(district):

    district_geom = district.geometry()

    district_area_km2 = (
        district_geom
        .area(GEOMETRY_ERROR_METERS)
        .divide(1e6)
    )

    intersecting_pas = panama_pas.filterBounds(
        district_geom
    )

    # Union the intersecting protected-area polygons first.
    # This prevents overlapping WDPA records from being
    # counted more than once in the same district.

    pa_union = intersecting_pas.geometry(
        GEOMETRY_ERROR_METERS
    )

    overlap_area_km2 = (
        pa_union
        .intersection(
            district_geom,
            ee.ErrorMargin(GEOMETRY_ERROR_METERS)
        )
        .area(GEOMETRY_ERROR_METERS)
        .divide(1e6)
    )

    protected_percentage = (
        overlap_area_km2
        .divide(district_area_km2)
        .multiply(100)
    )

    return district.set({
        "PROTECTED_AREA_KM2": overlap_area_km2,
        "PROTECTED_AREA_PCT": protected_percentage
    })


districts_ee = districts_ee.map(
    calculate_protected_area
)

### Extract GEE results into Pandas

In [ ]:
# 7. EXTRACT GEE RESULTS INTO PANDAS

print("\nExtracting district statistics from GEE...")


# The GEE features already carry the ANATI district identifiers.
# We retrieve their properties, then attach those properties
# back to the original ANATI GeoDataFrame.

gee_result = districts_ee.getInfo()

gee_records = [
    feature["properties"]
    for feature in gee_result["features"]
]

gee_df = pd.DataFrame(gee_records)


# Ensure that numeric properties are numeric.
numeric_columns = [
    "HEALTH_FACILITY_COUNT",
    "DISTRICT_AREA_KM2",
    "PROTECTED_AREA_KM2",
    "PROTECTED_AREA_PCT"
] + [
    f"AETI_{year}" for year in ANALYSIS_YEARS
]

for column in numeric_columns:

    if column in gee_df.columns:
        gee_df[column] = pd.to_numeric(
            gee_df[column],
            errors="coerce"
        )


# Process land-cover histograms

def process_lulc_histogram(histogram):

    if not isinstance(histogram, dict):
        return {
            "DOMINANT_LANDUSE": np.nan,
            "DOMINANT_LANDUSE_PCT": np.nan,
            "TOTAL_LULC_PIXELS": 0
        }

    counts = {}

    for key, value in histogram.items():

        try:
            class_code = int(float(key))
            pixel_count = float(value)

            counts[class_code] = pixel_count

        except (ValueError, TypeError):
            continue

    if not counts:

        return {
            "DOMINANT_LANDUSE": np.nan,
            "DOMINANT_LANDUSE_PCT": np.nan,
            "TOTAL_LULC_PIXELS": 0
        }

    total_pixels = sum(counts.values())

    dominant_code = max(
        counts,
        key=counts.get
    )

    dominant_count = counts[dominant_code]

    dominant_label = class_labels.get(
        dominant_code,
        f"Class {dominant_code}"
    )

    dominant_percentage = (
        dominant_count / total_pixels * 100
        if total_pixels > 0
        else np.nan
    )

    return {
        "DOMINANT_LANDUSE": dominant_label,
        "DOMINANT_LANDUSE_PCT": dominant_percentage,
        "TOTAL_LULC_PIXELS": total_pixels
    }


lulc_processed = gee_df[
    "LULC_HISTOGRAM"
].apply(process_lulc_histogram).apply(pd.Series)


gee_df = pd.concat(
    [
        gee_df.drop(columns=["LULC_HISTOGRAM"]),
        lulc_processed
    ],
    axis=1
)


# Calculate AETI summary metrics

aeti_columns = [
    f"AETI_{year}" for year in ANALYSIS_YEARS
]


def calculate_aeti_summary(row):

    values = [
        safe_float(row.get(column))
        for column in aeti_columns
    ]

    values = np.array(values, dtype=float)

    result = {
        "AVG_AETI_2021_2023_MM": np.nan,
        "AETI_CHANGE_2021_2023": np.nan,
        "AETI_TREND_2021_2023": np.nan
    }

    # Require all three years for a comparable 3-year summary.
    if not np.all(np.isfinite(values)):
        return pd.Series(result)

    result["AVG_AETI_2021_2023_MM"] = values.mean()

    result["AETI_CHANGE_2021_2023"] = (
        values[2] - values[0]
    )

    result["AETI_TREND_2021_2023"] = np.polyfit(
        ANALYSIS_YEARS,
        values,
        1
    )[0]

    return pd.Series(result)


aeti_summary = gee_df.apply(
    calculate_aeti_summary,
    axis=1
)

gee_df = pd.concat(
    [gee_df, aeti_summary],
    axis=1
)


# Join GEE statistics to ANATI

df = districts_gdf[
    [
        "DISTRICT_CODE",
        "DISTRICT",
        "PROVINCE",
        "ADMIN_KEY"
    ]
].merge(
    gee_df,
    on=["DISTRICT_CODE", "DISTRICT", "PROVINCE"],
    how="left",
    validate="one_to_one"
)

### Population data

In [ ]:
# 8. LOAD POPULATION DATA

print("\nLoading population data...")


population = read_admin_csv(
    POPULATION_FILE,
    "Population data"
)


population_columns = [
    "POPULATION_2021",
    "POPULATION_2022",
    "POPULATION_2023"
]

missing = [
    column for column in population_columns
    if column not in population.columns
]

if missing:
    raise ValueError(
        f"Population file is missing columns: {missing}"
    )


report_unmatched(
    population,
    districts_gdf,
    "Population data"
)


df = df.merge(
    population[
        ["ADMIN_KEY"] + population_columns
    ],
    on="ADMIN_KEY",
    how="left",
    validate="one_to_one"
)


for column in population_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    df.loc[df[column] <= 0, column] = np.nan


# Calculate health facility rate

df["HEALTH_FACILITY_COUNT"] = pd.to_numeric(
    df["HEALTH_FACILITY_COUNT"],
    errors="coerce"
)

df["HEALTH_FACILITIES_PER_10000"] = (
    df["HEALTH_FACILITY_COUNT"]
    / df["POPULATION_2021"]
    * 10000
)

df["HEALTH_FACILITIES_PER_100KM2"] = (
    df["HEALTH_FACILITY_COUNT"]
    / df["DISTRICT_AREA_KM2"]
    * 100
)

### Purchasing power

In [ ]:
# 9. LOAD ESRI PURCHASING POWER DATA

print("\nLoading purchasing power data...")


purchasing_power = read_admin_csv(
    PURCHASING_POWER_FILE,
    "Purchasing power data"
)


# Expected standardized fields.
# Rename source columns before running if necessary.

if "PURCHASING_POWER" not in purchasing_power.columns:
    raise ValueError(
        "Purchasing power file must contain "
        "PURCHASING_POWER."
    )


if "PURCHASING_POWER_INDEX" not in purchasing_power.columns:

    purchasing_power["PURCHASING_POWER_INDEX"] = np.nan


report_unmatched(
    purchasing_power,
    districts_gdf,
    "Purchasing power data"
)


df = df.merge(
    purchasing_power[
        [
            "ADMIN_KEY",
            "PURCHASING_POWER",
            "PURCHASING_POWER_INDEX"
        ]
    ],
    on="ADMIN_KEY",
    how="left",
    validate="one_to_one"
)


df["PURCHASING_POWER"] = pd.to_numeric(
    df["PURCHASING_POWER"],
    errors="coerce"
)

df["PURCHASING_POWER_INDEX"] = pd.to_numeric(
    df["PURCHASING_POWER_INDEX"],
    errors="coerce"
)

### Mortality data

In [ ]:
# 10. LOAD INEC MORTALITY DATA

print("\nLoading mortality data...")


mortality = read_admin_csv(
    MORTALITY_FILE,
    "Mortality data"
)


rate_columns = [
    f"MORTALITY_RATE_{year}"
    for year in ANALYSIS_YEARS
]

death_columns = [
    f"DEATHS_{year}"
    for year in ANALYSIS_YEARS
]


# Published rates are preferred if available.
# Death counts are retained to calculate a pooled rate
# when all three annual counts and population denominators exist.

for column in rate_columns + death_columns:

    if column not in mortality.columns:
        mortality[column] = np.nan

    mortality[column] = pd.to_numeric(
        mortality[column],
        errors="coerce"
    )


report_unmatched(
    mortality,
    districts_gdf,
    "Mortality data"
)


df = df.merge(
    mortality[
        ["ADMIN_KEY"] + rate_columns + death_columns
    ],
    on="ADMIN_KEY",
    how="left",
    validate="one_to_one"
)


# Preferred outcome:
# mean published annual mortality rate

# Require all three years for the 3-year average.
# This avoids comparing districts with different year coverage.

df["MORTALITY_RATE_2021_2023"] = (
    df[rate_columns].mean(axis=1)
)

complete_rate_rows = (
    df[rate_columns].notna().all(axis=1)
)

df.loc[
    ~complete_rate_rows,
    "MORTALITY_RATE_2021_2023"
] = np.nan


# Alternative outcome:
# pooled deaths / pooled population

df["TOTAL_DEATHS_2021_2023"] = (
    df[death_columns].sum(
        axis=1,
        min_count=3
    )
)

population_total = df[
    population_columns
].sum(
    axis=1,
    min_count=3
)

df["POOLED_MORTALITY_RATE_2021_2023"] = (
    df["TOTAL_DEATHS_2021_2023"]
    / population_total
    * 1000
)


# If published rates are unavailable for a district,
# use the pooled rate as a fallback, provided the necessary
# counts and denominators are complete.

df["MORTALITY_RATE_2021_2023"] = (
    df["MORTALITY_RATE_2021_2023"]
    .fillna(
        df["POOLED_MORTALITY_RATE_2021_2023"]
        * (MORTALITY_RATE_MULTIPLIER / 1000)
    )
)

### Final dataframe

In [ ]:
# 11. FINAL ANALYTICAL DATAFRAME

print("\nPreparing final dataframe...")


# Primary five predictors and outcome:
primary_columns = [
    "HEALTH_FACILITIES_PER_10000",
    "DOMINANT_LANDUSE",
    "AETI_TREND_2021_2023",
    "PURCHASING_POWER",
    "PROTECTED_AREA_PCT",
    "MORTALITY_RATE_2021_2023"
]


# Additional useful supporting variables:
supporting_columns = [
    "DISTRICT_CODE",
    "DISTRICT",
    "PROVINCE",
    "ADMIN_KEY",

    "HEALTH_FACILITY_COUNT",
    "HEALTH_FACILITIES_PER_100KM2",
    "DISTRICT_AREA_KM2",

    "DOMINANT_LANDUSE_PCT",
    "TOTAL_LULC_PIXELS",

    "AETI_2021",
    "AETI_2022",
    "AETI_2023",
    "AVG_AETI_2021_2023_MM",
    "AETI_CHANGE_2021_2023",

    "PROTECTED_AREA_KM2",

    "POPULATION_2021",
    "POPULATION_2022",
    "POPULATION_2023",

    "PURCHASING_POWER_INDEX",

    "MORTALITY_RATE_2021",
    "MORTALITY_RATE_2022",
    "MORTALITY_RATE_2023",

    "DEATHS_2021",
    "DEATHS_2022",
    "DEATHS_2023",

    "TOTAL_DEATHS_2021_2023",
    "POOLED_MORTALITY_RATE_2021_2023"
]


final_columns = list(dict.fromkeys(
    supporting_columns + primary_columns
))


final_columns = [
    column for column in final_columns
    if column in df.columns
]


df = df[final_columns].copy()


# Sort rows consistently

df = df.sort_values(
    ["PROVINCE", "DISTRICT"]
).reset_index(drop=True)


# Check that each ANATI
# district appears exactly once

assert df["ADMIN_KEY"].is_unique, (
    "The final dataframe contains duplicate district keys."
)

assert len(df) == len(districts_gdf), (
    "The final dataframe does not have one row per ANATI district."
)

### Quality control

In [ ]:
# 12. QUALITY CONTROL

print("\n==============================")
print("FINAL DATAFRAME QUALITY CHECK")
print("==============================")


print("\nNumber of ANATI districts:", len(districts_gdf))

print("Number of final dataframe rows:", len(df))

print("\nMissing values by column:")

print(
    df.isna().sum().sort_values(
        ascending=False
    ).to_string()
)


print("\nPrimary variable summary:")

print(
    df[primary_columns].describe(
        include="all"
    ).to_string()
)


print("\nDistricts with no matched population:")

print(
    df.loc[
        df["POPULATION_2021"].isna(),
        ["PROVINCE", "DISTRICT"]
    ].to_string(index=False)
)


print("\nDistricts with no mortality outcome:")

print(
    df.loc[
        df["MORTALITY_RATE_2021_2023"].isna(),
        ["PROVINCE", "DISTRICT"]
    ].to_string(index=False)
)


print("\nProtected area percentages above 100%:")

print(
    df.loc[
        df["PROTECTED_AREA_PCT"] > 100,
        [
            "PROVINCE",
            "DISTRICT",
            "PROTECTED_AREA_PCT"
        ]
    ].to_string(index=False)
)


print("\nDominant land-cover classes:")

print(
    df["DOMINANT_LANDUSE"].value_counts(
        dropna=False
    ).to_string()
)

### Exports

In [ ]:
# 13. EXPORT CSV AND GEOPACKAGE

print("\nExporting final files...")


# Tabular dataframe
df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


# Spatial dataframe:
# Attach the exact same district statistics to the original
# ANATI geometries. The geometry standard never changes.

final_gdf = districts_gdf.merge(
    df.drop(
        columns=[
            "DISTRICT",
            "PROVINCE"
        ],
        errors="ignore"
    ),
    on=[
        "DISTRICT_CODE",
        "ADMIN_KEY"
    ],
    how="left",
    validate="one_to_one"
)


final_gdf = gpd.GeoDataFrame(
    final_gdf,
    geometry="geometry",
    crs="EPSG:4326"
)


final_gdf.to_file(
    OUTPUT_GPKG,
    layer="panama_one_health_districts",
    driver="GPKG"
)


print("\nFinished.")

print("CSV:", OUTPUT_CSV)
print("GeoPackage:", OUTPUT_GPKG)

print("\nFinal dataframe shape:", df.shape)

print("\nFinal dataframe columns:")
print(df.columns.tolist())